# OR · 07 Safety Stock Intro


## 1️⃣ Configuración del Entorno

## 🎯 Objetivos de Aprendizaje

- Definir qué aprenderá el lector (máx. 5–7 puntos).
- Conectar con el caso de uso del dominio (demanda, logística, IoT).
- Incluir resultados verificables (métricas, validaciones, artefactos generados).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path

# Rutas
DATA_DIR = Path("../../data/raw")
OUTPUT_DIR = Path("../../data/processed")
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

## 2️⃣ Cargar y Preparar Datos

In [ ]:
# Cargar datos
df_orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=['order_date'])
df_products = pd.read_csv(DATA_DIR / "products.csv")
df_inventory = pd.read_csv(DATA_DIR / "inventory.csv")

print("📊 Datos cargados:")
print(f"  - Órdenes: {len(df_orders)} registros")
print(f"  - Productos: {len(df_products)} SKUs")
print(f"  - Inventario: {len(df_inventory)} registros")

display(df_orders.head(3))

## 3️⃣ Agregación de Demanda Diaria

In [ ]:
# Agregar demanda diaria por SKU
df_daily_demand = df_orders.groupby(['sku', 'order_date'])['quantity'].sum().reset_index()
df_daily_demand.rename(columns={'quantity': 'daily_demand'}, inplace=True)

print("📅 Demanda diaria agregada")
print(f"Total registros: {len(df_daily_demand)}")
display(df_daily_demand.head())

# Distribución de demanda para un SKU ejemplo
sample_sku = df_daily_demand['sku'].iloc[0]
sample_data = df_daily_demand[df_daily_demand['sku'] == sample_sku]

fig = px.histogram(
    sample_data, x='daily_demand', 
    title=f"Distribución de Demanda Diaria - {sample_sku}",
    labels={'daily_demand': 'Demanda Diaria', 'count': 'Frecuencia'},
    nbins=20
)
fig.show()

## 4️⃣ Cálculo de Variabilidad de Demanda

In [ ]:
# Calcular estadísticas de demanda por SKU
demand_stats = df_daily_demand.groupby('sku')['daily_demand'].agg([
    ('avg_demand', 'mean'),
    ('std_demand', 'std'),
    ('cv', lambda x: x.std() / x.mean() if x.mean() > 0 else 0)  # Coeficiente de variación
]).reset_index()

# Enriquecer con nombres de producto
demand_stats = demand_stats.merge(df_products[['sku', 'product_name', 'category']], on='sku')

print("📊 Estadísticas de Demanda:")
display(demand_stats.head())
print(f"\n📈 Promedio CV (variabilidad): {demand_stats['cv'].mean():.2f}")

## 5️⃣ Parámetros de Política de Inventario

In [ ]:
# Parámetros de negocio
SERVICE_LEVEL = 0.95  # 95% nivel de servicio
LEAD_TIME_DAYS = 7    # 7 días lead time de reabastecimiento

# Z-score para nivel de servicio
z_score = stats.norm.ppf(SERVICE_LEVEL)

print(f"🎯 Parámetros de Política:")
print(f"  - Nivel de servicio: {SERVICE_LEVEL*100:.0f}%")
print(f"  - Z-score: {z_score:.2f}")
print(f"  - Lead time: {LEAD_TIME_DAYS} días")
print(f"\n📚 Interpretación: Con 95% de servicio, tenemos 5% de probabilidad de stockout")

## 6️⃣ Fórmula de Stock de Seguridad

### Fórmula Clásica:
```
Safety Stock = Z × σ_demand × √(Lead Time)
```

Donde:
- **Z**: Z-score del nivel de servicio (1.65 para 95%)
- **σ_demand**: Desviación estándar de la demanda diaria
- **Lead Time**: Tiempo de reabastecimiento en días

In [ ]:
# Calcular stock de seguridad
demand_stats['safety_stock'] = (
    z_score * demand_stats['std_demand'] * np.sqrt(LEAD_TIME_DAYS)
).round(0).astype(int)

# Calcular punto de reorden (ROP = demanda durante lead time + safety stock)
demand_stats['reorder_point'] = (
    (demand_stats['avg_demand'] * LEAD_TIME_DAYS) + demand_stats['safety_stock']
).round(0).astype(int)

# Días de cobertura del safety stock
demand_stats['coverage_days'] = (
    demand_stats['safety_stock'] / demand_stats['avg_demand']
).round(1)

print("🛡️  Stock de Seguridad Calculado:")
display(demand_stats[[
    'sku', 'product_name', 'avg_demand', 'std_demand', 
    'safety_stock', 'reorder_point', 'coverage_days'
]])

## 7️⃣ Análisis de Cobertura

In [ ]:
# Top 10 productos con mayor safety stock
top_safety = demand_stats.nlargest(10, 'safety_stock')

fig = px.bar(
    top_safety,
    x='product_name',
    y='safety_stock',
    color='category',
    title="Top 10 Productos por Stock de Seguridad",
    labels={'safety_stock': 'Safety Stock (unidades)', 'product_name': 'Producto'}
)
fig.update_xaxis(tickangle=-45)
fig.show()

# Distribución de días de cobertura
fig2 = px.histogram(
    demand_stats, x='coverage_days',
    title="Distribución de Días de Cobertura del Safety Stock",
    labels={'coverage_days': 'Días de Cobertura'},
    nbins=20
)
fig2.show()

print(f"📊 Cobertura promedio: {demand_stats['coverage_days'].mean():.1f} días")
print(f"📊 Cobertura mediana: {demand_stats['coverage_days'].median():.1f} días")

## 8️⃣ Comparar con Inventario Actual

In [ ]:
# Unir con inventario actual
inventory_comparison = demand_stats.merge(
    df_inventory.groupby('sku')['quantity'].sum().reset_index(),
    on='sku',
    how='left'
)
inventory_comparison.rename(columns={'quantity': 'current_stock'}, inplace=True)
inventory_comparison['current_stock'].fillna(0, inplace=True)

# Identificar productos con stock insuficiente
inventory_comparison['needs_replenishment'] = (
    inventory_comparison['current_stock'] < inventory_comparison['reorder_point']
)

# Gap de inventario
inventory_comparison['stock_gap'] = (
    inventory_comparison['reorder_point'] - inventory_comparison['current_stock']
).clip(lower=0)

print("🔍 Comparación con Inventario Actual:")
display(inventory_comparison[[
    'sku', 'product_name', 'current_stock', 'safety_stock', 
    'reorder_point', 'needs_replenishment', 'stock_gap'
]].head(10))

# Resumen
need_replen = inventory_comparison['needs_replenishment'].sum()
print(f"\n⚠️  Productos que necesitan reabastecimiento: {need_replen} de {len(inventory_comparison)}")
print(f"📦 Gap total de inventario: {inventory_comparison['stock_gap'].sum():.0f} unidades")

## 9️⃣ Sensibilidad del Nivel de Servicio

In [ ]:
# Analizar diferentes niveles de servicio
service_levels = [0.90, 0.95, 0.98, 0.99]
sample_sku_data = demand_stats.iloc[0]

sensitivity_results = []
for sl in service_levels:
    z = stats.norm.ppf(sl)
    ss = z * sample_sku_data['std_demand'] * np.sqrt(LEAD_TIME_DAYS)
    sensitivity_results.append({
        'service_level': f"{sl*100:.0f}%",
        'z_score': z,
        'safety_stock': int(ss)
    })

df_sensitivity = pd.DataFrame(sensitivity_results)

fig = px.bar(
    df_sensitivity,
    x='service_level',
    y='safety_stock',
    title=f"Sensibilidad del Safety Stock al Nivel de Servicio<br>SKU: {sample_sku_data['sku']}",
    labels={'service_level': 'Nivel de Servicio', 'safety_stock': 'Safety Stock (unidades)'},
    text='safety_stock'
)
fig.update_traces(textposition='outside')
fig.show()

print("📊 Análisis de Sensibilidad:")
display(df_sensitivity)

## 🔟 Guardar Resultados

In [ ]:
# Guardar políticas de inventario
output_file = OUTPUT_DIR / "inventory_policies.csv"
inventory_comparison.to_csv(output_file, index=False)

print(f"💾 Políticas guardadas: {output_file}")
print(f"📏 Dimensiones: {inventory_comparison.shape}")

# Resumen ejecutivo
summary = {
    'total_skus': len(inventory_comparison),
    'avg_safety_stock': inventory_comparison['safety_stock'].mean(),
    'total_safety_stock': inventory_comparison['safety_stock'].sum(),
    'skus_need_replenishment': need_replen,
    'total_stock_gap': inventory_comparison['stock_gap'].sum(),
    'service_level': SERVICE_LEVEL,
    'lead_time_days': LEAD_TIME_DAYS
}

print("\n📋 RESUMEN EJECUTIVO")
print("="*50)
for key, value in summary.items():
    print(f"  {key}: {value}")

## 🎓 Conclusiones

**Aprendizajes Clave:**
1. ✅ **Variabilidad de Demanda**: La desviación estándar mide incertidumbre
2. ✅ **Nivel de Servicio**: Trade-off entre costo y servicio (95% → z=1.65)
3. ✅ **Fórmula Clásica**: `SS = Z × σ × √LT` captura incertidumbre
4. ✅ **Punto de Reorden**: `ROP = Demanda_LT + SS` dispara reabastecimiento

**Decisiones de Negocio:**
- 📊 Productos con alta variabilidad (CV > 0.5) requieren más safety stock
- 💰 Aumentar servicio de 95% a 99% incrementa SS en ~40%
- 🎯 Políticas diferenciadas por categoría ABC optimizan capital de trabajo

**Próximos Pasos:**
- Implementar políticas (Q, R) con tamaño de lote económico (EOQ)
- Incluir variabilidad de lead time (ver OR-01)
- Multi-echelon inventory optimization (ver OR-04)

---

**🔗 Notebooks Relacionados:**
- [OR-01: Stock de Seguridad](../50_optimization_or/OR-01-stock_seguridad.ipynb) - Versión avanzada
- [OR-02: Políticas de Inventario](../50_optimization_or/OR-02-politicas_inventario.ipynb)
- [BA-01: Dashboard OTIF](../40_business_analytics_bi/BA-01-dashboard_otif.ipynb)

## 🛠️ Funciones Reutilizables

In [ ]:
def calculate_safety_stock(
    avg_demand: float,
    std_demand: float,
    service_level: float,
    lead_time_days: int
) -> dict:
    """
    Calcula stock de seguridad y punto de reorden.
    
    Args:
        avg_demand: Demanda promedio diaria
        std_demand: Desviación estándar de demanda diaria
        service_level: Nivel de servicio deseado (0-1)
        lead_time_days: Lead time de reabastecimiento (días)
    
    Returns:
        Dict con safety_stock, reorder_point, z_score
    """
    z_score = stats.norm.ppf(service_level)
    safety_stock = z_score * std_demand * np.sqrt(lead_time_days)
    reorder_point = (avg_demand * lead_time_days) + safety_stock
    
    return {
        'safety_stock': int(safety_stock),
        'reorder_point': int(reorder_point),
        'z_score': round(z_score, 2),
        'coverage_days': round(safety_stock / avg_demand, 1) if avg_demand > 0 else 0
    }

# Ejemplo de uso:
# result = calculate_safety_stock(avg_demand=50, std_demand=15, service_level=0.95, lead_time_days=7)
# print(result)

## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Consideraciones de almacenamiento/cómputo/visualización.

**Retención**
- Política por zonas (raw/curated/analytics) y ventanas temporales.

**Gobernanza**
- Calidad de datos, seguridad/PII, linaje, versionado de modelos/artefactos.